In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from typing import TypedDict, Annotated
from PIL import Image
import random
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
# from langchain_mcp_adapters.client import MultiServerMCPClient
# from deepagents import create_deep_agent
# from deepagents.backends import FileSystemBackend
load_dotenv(override=True)

In [ ]:
# Simple agent creation
model = ChatGoogleGenerativeAI(
model="gemini-3.5-flash-lite"
)

agent = create_agent(model=model, system_prompt="You are a helful assistant")

result = agent.invoke({"messages": [{"role": "user", "content": "What is MCP in 2 sentences"}]})
print(result["messages"][-1].content)

In [ ]:
# Tools
@tool
def get_weather(city: str):
    """Return todays weather in a city"""
    pretend = {"London": "rainy, 14 degrees", "Rome": "sunny, 27 degrees"}
    return pretend.get(city, "clear, 20 degrees")

agent = create_agent(model=model, tools=[get_weather], system_prompt="You are a travel assitant")

result = agent.invoke({"messages": [{"role": "user", "content": "What is MCP in 2 sentences"}]})

In [ ]:
memory_agent = create_agent(model = model,
system_prompt="You are a helpful agent",
tools=[get_weather],
checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "trip-planning"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "I am planning a trip to London"}]}, config=config)
result = memory_agent.invoke({"messages": [{"role": "user", "content": "What is weather of trip where I am going"}]}, config=config)
print(result["messages"][-1].content[0]["text"])

## Structured outputs for langchain

In [ ]:
class CityReport(BaseModel):
    city:str = Field(description="The city name")
    weather:str = Field(description="A short weather desciption")
    population:str = Field(description="The population")

report_agent = create_agent(
    model=model,
    tools = [get_weather],
    response_format=CityReport # allows for structured output
)


## Middleware

In [ ]:
@wrap_tool_call
def log_tool_calls(request, handler):
    call = request.tool_call
    print(f"  [middleware] calling {call['name']} with {call['args']}")
    return handler(request)

watched_agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful agent",
    middleware=[log_tool_calls]
)

result = watched_agent.invoke({"messages": [{"role":"user", "content": "Weather and population of London and Rome?"}]})
print("\nFinal answer: ", result["messages"][-1].content[0]["text"])

## MCP for playwright and using the browser to do stuff

In [ ]:
import sys

if sys.platform=="win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied windows adjustment")

In [ ]:
from langchain.tools import tool
from playwright.async_api import async_playwright

@tool
async def get_top_bbc_news():
    """Get top BBC headlines"""

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page()
        await page.goto("https://www.bbc.com/news")

        headlines = await page.locator("h2").all_text_contents()

        await browser.close()

        return headlines[:3]

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_top_bbc_news]
)

In [ ]:
# async for chunk in agent.astream(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "What are the top 3 BBC headlines right now?"
#             }
#         ]
#     },
#     stream_mode="updates"
# ):
#     print(chunk)
## SSL error for corporate proxy

## Ok so a research agent that can plan search and write files

In [ ]:
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

researcher = create_deep_agent(model = model, tools=[search],
                                system_prompt = "You are a research agent plan your work with the given research tools reseaerch with search tool and write your findings into a file as a tidy markdown",
                                backend=FileSystemBackend(root_dir=sandbox, virtual_mode=True)
)

In [ ]:
brief = """Our company is planning to move its sales fleet into electric vehicales 
find the best electric vehicles around me in Pune, India and the best
charging network provider companies. Write a one page  markdown briefing with a heading and short section
for each to the file charging.md"""

result = researcher.invoke({"messages": [{"role": "user", "content": brief}]})
print(result["messages"][-1].content[0]["text"])